In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.3 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO

import os
import shutil
import random
from pathlib import Path
import yaml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_ROOT = "/content/drive/MyDrive/thermal project/Thermal-RGB-Integrated-Detection-with-Encoded-Noise-Tolerance"

YOLO_ROOT = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "flir",
    "flir_yolo"
)

TRAIN_IMAGES = os.path.join(YOLO_ROOT, "images", "train")
TRAIN_LABELS = os.path.join(YOLO_ROOT, "labels", "train")

VAL_IMAGES = os.path.join(YOLO_ROOT, "images", "val")
VAL_LABELS = os.path.join(YOLO_ROOT, "labels", "val")

In [5]:
SUBSET_ROOT = os.path.join(YOLO_ROOT, "subset")

folders = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for folder in folders:
    Path(os.path.join(SUBSET_ROOT, folder)).mkdir(parents=True, exist_ok=True)

In [6]:
SEED = 42
NUM_IMAGES = 5000

random.seed(SEED)

all_images = sorted(os.listdir(TRAIN_IMAGES))

selected_images = random.sample(all_images, NUM_IMAGES)

print(f"Selected {len(selected_images)} images")

Selected 5000 images


In [7]:
import os
import random

SEED = 42
NUM_IMAGES = 5000

random.seed(SEED)

train_images = sorted([
    os.path.join(TRAIN_IMAGES, f)
    for f in os.listdir(TRAIN_IMAGES)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

selected = random.sample(train_images, NUM_IMAGES)

train_txt = os.path.join(YOLO_ROOT, "train_subset.txt")

with open(train_txt, "w") as f:
    f.write("\n".join(selected))

print(f"Created {train_txt}")
print(f"Images selected: {len(selected)}")

Created /content/drive/MyDrive/thermal project/Thermal-RGB-Integrated-Detection-with-Encoded-Noise-Tolerance/datasets/flir/flir_yolo/train_subset.txt
Images selected: 5000


In [8]:
val_images = sorted([
    os.path.join(VAL_IMAGES, f)
    for f in os.listdir(VAL_IMAGES)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

val_txt = os.path.join(YOLO_ROOT, "val.txt")

with open(val_txt, "w") as f:
    f.write("\n".join(val_images))

print(f"Created {val_txt}")
print(f"Validation images: {len(val_images)}")

Created /content/drive/MyDrive/thermal project/Thermal-RGB-Integrated-Detection-with-Encoded-Noise-Tolerance/datasets/flir/flir_yolo/val.txt
Validation images: 1144


In [9]:
import yaml

data = {
    "train": train_txt,
    "val": val_txt,
    "nc": 1,
    "names": ["person"]
}

yaml_path = os.path.join(YOLO_ROOT, "data_subset.yaml")

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Created:", yaml_path)

Created: /content/drive/MyDrive/thermal project/Thermal-RGB-Integrated-Detection-with-Encoded-Noise-Tolerance/datasets/flir/flir_yolo/data_subset.yaml


In [10]:
print("Training Images :", len(os.listdir(os.path.join(SUBSET_ROOT, "images/train"))))
print("Training Labels :", len(os.listdir(os.path.join(SUBSET_ROOT, "labels/train"))))

print()

print("Validation Images :", len(os.listdir(os.path.join(SUBSET_ROOT, "images/val"))))
print("Validation Labels :", len(os.listdir(os.path.join(SUBSET_ROOT, "labels/val"))))

Training Images : 4184
Training Labels : 4184

Validation Images : 0
Validation Labels : 0


In [11]:
model = YOLO("yolov8n.pt")

In [12]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,
    project=os.path.join(PROJECT_ROOT, "runs"),
    name="thermal_yolov8n_subset",
    pretrained=True,
    exist_ok=True,
    save=True,
    val=True
)

Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/thermal project/Thermal-RGB-Integrated-Detection-with-Encoded-Noise-Tolerance/datasets/flir/flir_yolo/data_subset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale

In [13]:
RUN_DIR = os.path.join(
    PROJECT_ROOT,
    "runs",
    "thermal_yolov8n_subset"
)

print("Run Directory:")
print(RUN_DIR)

print("\nBest Model:")
print(os.path.exists(os.path.join(RUN_DIR, "weights", "best.pt")))

print("\nLast Model:")
print(os.path.exists(os.path.join(RUN_DIR, "weights", "last.pt")))

Run Directory:
/content/drive/MyDrive/thermal project/Thermal-RGB-Integrated-Detection-with-Encoded-Noise-Tolerance/runs/thermal_yolov8n_subset

Best Model:
True

Last Model:
True
